# 07b · Instance Segmentation, Panoptic Quality & SAM `[local]` (fine-tuning section: `[kaggle]`)

**Goal:** separate masks for every object. You'll see how **Mask R-CNN** extends the detector from 06, measure instance masks properly (**mask AP**, **panoptic quality**), and use **SAM 2**, a model that segments *anything* from a click or a box, including a detect-then-segment pipeline.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Instance vs. panoptic; Mask R-CNN = Faster R-CNN + a mask head; promptable segmentation |
| 📐 Math | RoIAlign, the mask loss, mask AP, Panoptic Quality = SQ × RQ |
| 🔨 From scratch | RoIAlign vs. RoIPool, mask IoU, PQ on a toy example, resolving overlapping masks |
| 🚀 Mask R-CNN | Pretrained on Penn-Fudan; fine-tuning (Kaggle) |
| 🚀 SAM 2 | Point and box prompts; YOLO → SAM detect-then-segment |
| 📊 Evaluate | Mask AP and PQ for Mask R-CNN vs. YOLO + SAM; mask quality given perfect boxes; speed |
| 🧪 Experiment | Where you click, negative clicks, sloppy boxes |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
from PIL import Image
from torchvision.ops import roi_align, roi_pool
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from cvlib.datasets import pennfudan
from cvlib.detection import mask_iou, coco_mask_ap, box_iou
from cvlib.metrics import panoptic_quality
from cvlib.data import model_path

os.environ["YOLO_VERBOSE"] = "False"
device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"
rng = np.random.default_rng(0)

---
# 🧠 1 · Intuition

### Instance segmentation = detection + a mask per box
Semantic segmentation (07a) colors every person pixel "person", and two people hugging become one blob. **Instance** segmentation outputs **one mask per object**: person #1, person #2. Counting, tracking and measuring individual objects need this.

### Mask R-CNN
Take Faster R-CNN (06) and add a third head in stage 2: for each detected box, a small FCN predicts a **28×28 mask** inside that box, which is then resized to the box. Detection decides *which objects*; the mask head decides *which pixels* within each one.

### Panoptic segmentation
Label every pixel, **and** separate the countable objects: "**things**" (people, cars: get instance IDs) vs. "**stuff**" (sky, road, grass: amorphous, no instances). No overlaps allowed: each pixel belongs to exactly one segment. Its metric, **PQ**, combines "did you find the segments?" with "how well did you outline them?".

### SAM: promptable segmentation
The **Segment Anything Model** (Meta, 2023; SAM 2 in 2024) was trained on over 1 billion masks to answer one question: *"given this prompt, what's the object?"* A prompt can be a **click** (point), a few clicks (positive and negative), or a **box**. It has no class labels: it segments *whatever* you point at.

Architecture: a heavy **image encoder** runs once per image → a light **prompt encoder** + **mask decoder** run per prompt in milliseconds. That's what makes interactive "click to cut out" tools feel instant. SAM 2 adds a memory so it can also follow an object through video.

**Detect-then-segment:** a detector (which knows classes) gives boxes; SAM turns each box into a precise mask. Instance segmentation without training a mask model.

---
# 📐 2 · The math

### RoIAlign vs. RoIPool
Stage 2 needs a fixed-size feature grid (e.g. 7×7 or 14×14) for each box, but boxes don't line up with feature-map cells (a box edge at x = 13.6 on a stride-16 map sits at cell 0.85).
- **RoIPool** (Fast R-CNN) **rounds** box coordinates to whole cells, then max-pools: up to half a cell of misalignment, i.e. 8 pixels at stride 16. Fine for classification, bad for pixel-accurate masks.
- **RoIAlign** (Mask R-CNN) keeps the exact fractional coordinates and **bilinearly interpolates** the feature map at sampling points inside each bin. No rounding anywhere.

### The mask loss
For each positive RoI with class $c$, the mask head outputs $K$ masks (one per class), but **only the mask of class $c$** is trained, with per-pixel **binary** cross-entropy against the ground-truth mask cropped to the box and resized to 28×28. Classes don't compete over pixels: the box classifier already decided the class.

### Mask AP
Exactly the AP/mAP recipe from 06a, with **mask IoU** (overlap of pixel sets) instead of box IoU. Same 0.50:0.95 thresholds, same 101-point interpolation.

### Panoptic Quality
Match predicted to ground-truth segments of the same class when IoU > 0.5 (with non-overlapping segments, such matches are unique). Then:

$$\text{PQ} = \frac{\sum_{(p,g) \in TP} \text{IoU}(p, g)}{|TP| + \tfrac{1}{2}|FP| + \tfrac{1}{2}|FN|} = \underbrace{\frac{\sum_{TP} \text{IoU}}{|TP|}}_{\text{SQ: segmentation quality}} \times \underbrace{\frac{|TP|}{|TP| + \tfrac12|FP| + \tfrac12|FN|}}_{\text{RQ: recognition quality (an F1)}}$$

PQ is computed per class and averaged. SQ tells you how good your outlines are; RQ tells you how many segments you found without hallucinating.

---
# 🔨 3 · From scratch

### 3.1 · RoIAlign vs. RoIPool on a box that doesn't line up

In [ ]:
fmap = torch.zeros(1, 1, 16, 16)
fmap[0, 0, :, :] = torch.arange(16.0)                               # feature value = x index of the cell
box = torch.tensor([[0, 3.3, 2.0, 9.7, 10.0]])                      # (batch idx, x1, y1, x2, y2), fractional
pooled = roi_pool(fmap, box, output_size=(4, 4), spatial_scale=1.0)
aligned = roi_align(fmap, box, output_size=(4, 4), spatial_scale=1.0, sampling_ratio=2, aligned=True)
true_centers = 3.3 + (9.7 - 3.3) * (np.arange(4) + 0.5) / 4 - 0.5     # cell i covers x ∈ [i, i+1), its center is at i + 0.5
print("true bin centers (as cell values):", np.round(true_centers, 2))
print("RoIAlign (bilinear)              :", aligned[0, 0, 0].numpy().round(2))
print("RoIPool (rounded + max)          :", pooled[0, 0, 0].numpy().round(2))

RoIAlign reproduces the exact positions; RoIPool snaps to whole cells and takes the max, so its features are shifted and stretched. Invisible for classification, visible at mask edges.

### 3.2 · Mask IoU and Panoptic Quality on a toy example

In [ ]:
def disk(cx, cy, r, H=60, W=100):
    yy, xx = np.mgrid[:H, :W]
    return (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2

gt = np.stack([disk(20, 30, 12), disk(50, 30, 10), disk(80, 30, 8)])            # 3 objects
pred = np.stack([disk(22, 31, 12), disk(50, 30, 6), disk(80, 10, 6)])            # good, too small, wrong place
print("mask IoU matrix (rows = predictions, cols = ground truth):\n", mask_iou(pred, gt).round(2))
pq = panoptic_quality([gt], [pred])
print({k: round(v, 3) if isinstance(v, float) else v for k, v in pq.items()})

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, m, t in zip(axes, [gt, pred], ["ground truth", "prediction"]):
    ax.imshow(sum((i + 1) * mm.astype(int) for i, mm in enumerate(m)), cmap="tab10", vmin=0, vmax=9); ax.set_title(t); ax.axis("off")
plt.show()

Only the first prediction matches (IoU > 0.5): **TP = 1, FP = 2, FN = 2**. The "too small" disk overlaps its object, but with IoU 0.36, so it counts as **both** a false positive and a missed object: PQ punishes a bad outline twice. SQ says the one match is well outlined; RQ says most objects weren't recognized properly.

### 3.3 · Making instance predictions non-overlapping

Mask R-CNN predicts each object's mask independently, so masks can overlap. Panoptic output can't. Standard fix: paint masks from highest to lowest score; each pixel keeps its first (most confident) owner, and segments that lose most of their pixels are dropped.

In [ ]:
def to_non_overlapping(masks, scores, min_score=0.5, min_keep=0.5):
    masks = np.asarray(masks, bool); scores = np.asarray(scores)
    shape = masks.shape[1:] if len(masks) else (1, 1)
    taken = np.zeros(shape, bool)
    out = []
    for i in np.argsort(-scores):
        if scores[i] < min_score:
            continue
        m = masks[i] & ~taken
        if m.sum() < min_keep * max(masks[i].sum(), 1):              # mostly hidden behind better segments: drop
            continue
        taken |= m; out.append(m)
    return np.stack(out) if out else np.zeros((0, *shape), bool)

---
# 🚀 4 · Mask R-CNN on Penn-Fudan pedestrians

Penn-Fudan has a separate mask for every pedestrian, perfect for instance segmentation. COCO's Mask R-CNN already knows "person", so start by scoring it with no training.

In [ ]:
items = pennfudan()
perm = rng.permutation(len(items))
train_items, test_items = [items[i] for i in perm[:120]], [items[i] for i in perm[120:]]

def load_gt(it):
    m = np.array(Image.open(it["mask"]))
    return np.stack([m == i for i in it["instance_ids"]])

def load_rgb(it):
    return np.array(Image.open(it["path"]).convert("RGB"))

gt_masks = [load_gt(it) for it in test_items]
print(f"test: {len(test_items)} images, {sum(len(g) for g in gt_masks)} pedestrians")

mrcnn = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT).eval().to(device)

@torch.no_grad()
def maskrcnn_predict(model, it, keep_label=1, score_thresh=0.05):
    model.roi_heads.score_thresh = score_thresh
    x = torchvision.transforms.functional.to_tensor(Image.open(it["path"]).convert("RGB")).to(device)
    d = model([x])[0]
    sel = (d["labels"] == keep_label).cpu().numpy()
    masks = (d["masks"][:, 0] > 0.5).cpu().numpy()[sel]              # soft masks → binary at 0.5
    return masks, d["scores"].cpu().numpy()[sel], d["boxes"].cpu().numpy()[sel]

t0 = time.perf_counter()
mr_out = [maskrcnn_predict(mrcnn, it) for it in test_items]
mr_ms = (time.perf_counter() - t0) / len(test_items) * 1000
print(f"Mask R-CNN (COCO, no fine-tuning): {mr_ms:.0f} ms/img on {device}")

def overlay(ax, it, masks, boxes=None):
    vis = load_rgb(it).astype(float)
    for k, mm in enumerate(masks):
        vis[mm] = 0.45 * vis[mm] + 0.55 * np.array(plt.cm.tab10(k % 10)[:3]) * 255
    vis = vis.astype(np.uint8)
    if boxes is not None:
        for bx in np.asarray(boxes).astype(int):
            cv2.rectangle(vis, tuple(bx[:2]), tuple(bx[2:]), (255, 255, 255), 2)
    ax.imshow(vis); ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(15, 4.6))
for ax, it, (m, s, b) in zip(axes, test_items[:4], mr_out[:4]):
    overlay(ax, it, m[s > 0.5])
plt.suptitle("Mask R-CNN instance masks (score > 0.5): one color per person"); plt.tight_layout(); plt.show()

### 4.1 · Fine-tuning Mask R-CNN `[kaggle]`

Same recipe as 06b, plus a new **mask predictor** head for 2 classes (background, person). On Kaggle this runs 10 epochs; locally a 1-epoch smoke test on a few images checks the code.

In [ ]:
class PennFudanMasks(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        img = torchvision.transforms.functional.to_tensor(Image.open(it["path"]).convert("RGB"))
        masks = torch.tensor(load_gt(it), dtype=torch.uint8)
        boxes = torch.tensor(it["boxes"], dtype=torch.float32)
        if self.train and torch.rand(1) < 0.5:                       # flip image, boxes AND masks together
            W = img.shape[-1]
            img, masks = img.flip(-1), masks.flip(-1)
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        return img, {"boxes": boxes, "labels": torch.ones(len(boxes), dtype=torch.int64), "masks": masks}

def finetune_maskrcnn(epochs, n_train=None, min_size=800):
    m = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT, min_size=min_size)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, 2)
    m.to(device)
    dl = torch.utils.data.DataLoader(PennFudanMasks(train_items[:n_train], True), batch_size=2, shuffle=True, collate_fn=lambda b: tuple(zip(*b)))
    opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=5e-4)
    for ep in range(epochs):
        m.train(); t0 = time.perf_counter(); tot = 0
        for imgs, tgts in dl:
            losses = m([i.to(device) for i in imgs], [{k: v.to(device) for k, v in t.items()} for t in tgts])
            loss = sum(losses.values()); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() / len(dl)
        print(f"epoch {ep + 1}: total loss {tot:.3f} (includes loss_mask) | {time.perf_counter() - t0:.0f}s")
    return m.eval()

ft_mrcnn = finetune_maskrcnn(epochs=10 if FULL else 1, n_train=None if FULL else 4, min_size=800 if FULL else 320)
ft_out = [maskrcnn_predict(ft_mrcnn, it) for it in test_items]
del ft_mrcnn

---
# 🚀 5 · SAM 2: segment anything from a prompt

We use **SAM 2.1 tiny** through Ultralytics (~39M parameters; larger variants are more accurate and slower). Ultralytics expects images as BGR arrays (OpenCV order).

### 5.1 · Point and box prompts

In [ ]:
from ultralytics import SAM, YOLO
sam = SAM(str(model_path("sam2.1_t.pt")))

it0 = test_items[0]
img = load_rgb(it0); bgr0 = img[:, :, ::-1].copy()
g = gt_masks[0][0]                                                     # person #1
ys, xs = np.nonzero(g)
cx, cy = int(np.median(xs)), int(np.median(ys))
x1, y1, x2, y2 = it0["boxes"][0].astype(int)

def sam_mask(bgr, **prompt):
    r = sam.predict(bgr, device=device, verbose=False, **prompt)[0]
    return r.masks.data.cpu().numpy().astype(bool)

prompts = {"1 click": dict(points=[[cx, cy]], labels=[1]), "box": dict(bboxes=[[x1, y1, x2, y2]])}
fig, axes = plt.subplots(1, len(prompts) + 1, figsize=(13, 4.6))
axes[0].imshow(img); axes[0].imshow(g, alpha=0.4, cmap="Greens"); axes[0].set_title("ground truth (person #1)")
for ax, (name, kw) in zip(axes[1:], prompts.items()):
    m = sam_mask(bgr0, **kw)[0]
    ax.imshow(img); ax.imshow(m, alpha=0.5, cmap="Reds")
    if "points" in kw: ax.plot(cx, cy, "g*", ms=18, mec="white")
    else: ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime", lw=2))
    ax.set_title(f"SAM 2 from {name}: IoU {mask_iou(m[None], g[None])[0, 0]:.2f}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

A single click is **ambiguous**: did you mean the shirt, the person, or the person + their bag? A box removes most of that ambiguity, which is why box prompts are the usual choice in pipelines. (Experiment 2 shows negative clicks as another fix.)

### 5.2 · Detect-then-segment: YOLO boxes → SAM masks

YOLO (a detector, knows classes) finds the people; SAM turns each box into a mask. Instance segmentation from two off-the-shelf models, with **no mask training at all**.

In [ ]:
yolo = YOLO(str(model_path("yolo11n.pt")))

def yolo_sam(it, conf=0.25):
    bgr = cv2.imread(str(it["path"]))
    det = yolo.predict(bgr, conf=conf, classes=[0], device=device, verbose=False)[0]   # class 0 = person
    boxes, scores = det.boxes.xyxy.cpu().numpy(), det.boxes.conf.cpu().numpy()
    if len(boxes) == 0:
        return np.zeros((0, *bgr.shape[:2]), bool), scores, boxes
    return sam_mask(bgr, bboxes=boxes.tolist()), scores, boxes

t0 = time.perf_counter()
ys_out = [yolo_sam(it) for it in test_items]
ys_ms = (time.perf_counter() - t0) / len(test_items) * 1000

fig, axes = plt.subplots(1, 4, figsize=(15, 4.6))
for ax, it, (m, s, b) in zip(axes, test_items[:4], ys_out[:4]):
    overlay(ax, it, m, b)
plt.suptitle("YOLO11n boxes (white) → SAM 2 masks"); plt.tight_layout(); plt.show()

---
# 📊 6 · Evaluate

### 6.1 · Mask AP and Panoptic Quality

In [ ]:
def report(name, outs, ms=None):
    ap = coco_mask_ap(gt_masks, [o[0] for o in outs], [o[1] for o in outs])
    pq = panoptic_quality(gt_masks, [to_non_overlapping(o[0], o[1]) for o in outs])
    t = f"{ms:6.0f}" if ms else "     -"
    print(f"{name:<32}{ap['mAP']:>8.3f}{ap['AP50']:>8.3f}{ap['AP75']:>8.3f}{pq['PQ']:>8.3f}{pq['SQ']:>8.3f}{pq['RQ']:>8.3f}{t:>9}")

print(f"{'model':<32}{'maskAP':>8}{'AP50':>8}{'AP75':>8}{'PQ':>8}{'SQ':>8}{'RQ':>8}{'ms/img':>9}")
report("Mask R-CNN (COCO)", mr_out, mr_ms)
report(f"Mask R-CNN fine-tuned ({'FULL' if FULL else 'SMOKE'})", ft_out)
report("YOLO11n → SAM 2.1-t", ys_out, ys_ms)

**Reading the table:**
- **Mask AP** ranks models across all confidence levels; **PQ** judges the final, non-overlapping output at a fixed operating point (score > 0.5). Report what matches your use case.
- **SQ vs. RQ** separates the two failure types: weak outlines vs. missed or hallucinated people.
- The **YOLO → SAM** pipeline, with zero mask training, actually **beats** COCO's Mask R-CNN here on both mask AP and PQ, and is faster on your Mac. Its ceiling is the detector: SAM can only segment what YOLO finds, and YOLO11n is a tiny detector.
- In smoke mode the "fine-tuned" row is meaningless (1 epoch, 4 images, new heads still random). On Kaggle it should improve over the COCO model.

### 6.2 · Mask quality alone: give every method the *true* boxes

Detection errors and mask errors are tangled together above. To isolate **mask quality**, prompt SAM with the **ground-truth boxes**, and compare with Mask R-CNN's masks for the detections that best match each true person.

In [ ]:
sam_ious, mr_ious = [], []
for it, gm, (mm, ss, bb) in zip(test_items, gt_masks, mr_out):
    sm = sam_mask(cv2.imread(str(it["path"])), bboxes=it["boxes"].tolist())
    sam_ious += list(np.diag(mask_iou(sm, gm)))
    if len(mm):
        best = box_iou(it["boxes"], bb).argmax(1)                         # Mask R-CNN detection closest to each true box
        mr_ious += list(np.diag(mask_iou(mm[best], gm)))
print(f"mean mask IoU given the true boxes → SAM 2.1-t: {np.mean(sam_ious):.3f} | Mask R-CNN (matched detections): {np.mean(mr_ious):.3f}")
plt.figure(figsize=(7, 3.2))
plt.hist([sam_ious, mr_ious], bins=20, range=(0, 1), label=["SAM (true box)", "Mask R-CNN"]); plt.xlabel("mask IoU per person")
plt.legend(); plt.grid(alpha=0.3); plt.show()

This separates "finding the person" from "outlining the person". Mask R-CNN predicts a 28×28 mask stretched over each box, which limits how sharp its outlines can be; SAM decodes masks at higher resolution, but is class-agnostic and can grab the wrong "thing" inside a box (a bag, a second person). In practice both are used: SAM for labeling tools, data generation and zero-shot pipelines; trained instance models where speed and fixed classes matter.

---
# 🧪 7 · Experiments: predict first, then run

**Experiment 1: where you click matters.** Click on the deepest interior point of person #1, near the edge of their silhouette, and on their head. Predict which gives the best mask.

In [ ]:
# Prediction: ...
dist = cv2.distanceTransform(g.astype(np.uint8), cv2.DIST_L2, 5)
inner = np.unravel_index(dist.argmax(), dist.shape)                    # (row, col) deepest inside the person
edge_pts = np.argwhere((dist > 0) & (dist < 3)); edge = edge_pts[len(edge_pts) // 2]
top_rows = ys < ys.min() + 20
head = (ys.min() + 10, int(np.median(xs[top_rows])))
for name, (py, px) in [("deepest interior point", inner), ("near the edge", edge), ("on the head", head)]:
    m = sam_mask(bgr0, points=[[int(px), int(py)]], labels=[1])[0]
    print(f"click {name:<24} → IoU {mask_iou(m[None], g[None])[0, 0]:.3f}")
# Compare with section 5.1, where a click on the shirt returned *just the shirt*. Here the deepest interior point
# and even the head gave the whole person, while a click near the edge is ambiguous (person? background? neighbor?).
# Results vary per image, which is why interactive tools let you add clicks rather than relying on one.

**Experiment 2: add a negative click.** Keep the positive click and add a *negative* one (label 0) on a region SAM wrongly included, or on the background next to the person. Predict the change.

In [ ]:
# Prediction: ...
m1 = sam_mask(bgr0, points=[[cx, cy]], labels=[1])[0]
wrong = np.argwhere(m1 & ~g)                                           # pixels SAM included but shouldn't have
if len(wrong) > 50:
    wy, wx = wrong[len(wrong) // 2]
else:
    wy, wx = cy, min(int(x2) + 15, img.shape[1] - 1)                  # just right of the person
m2 = sam_mask(bgr0, points=[[[cx, cy], [int(wx), int(wy)]]], labels=[[1, 0]])[0]
print(f"positive click only: IoU {mask_iou(m1[None], g[None])[0, 0]:.3f} | + negative click: IoU {mask_iou(m2[None], g[None])[0, 0]:.3f}")
# Negative clicks tell SAM "not this": the standard way interactive tools refine a mask in 2–3 clicks.

**Experiment 3: sloppy boxes.** Real detector boxes are imperfect. Shrink or grow the true boxes and predict how SAM's mask IoU changes.

In [ ]:
# Prediction: ...
for scale in (0.75, 0.9, 1.0, 1.1, 1.25):
    ious = []
    for it, gm in zip(test_items[:20], gt_masks[:20]):
        b = it["boxes"].copy(); c = (b[:, :2] + b[:, 2:]) / 2; wh = (b[:, 2:] - b[:, :2]) * scale
        sm = sam_mask(cv2.imread(str(it["path"])), bboxes=np.c_[c - wh / 2, c + wh / 2].tolist())
        ious += list(np.diag(mask_iou(sm, gm)))
    print(f"box scale {scale:>4}: mean mask IoU {np.mean(ious):.3f}")
# Asymmetric: boxes that are too small cut off heads and feet (SAM rarely segments far outside the box) and
# IoU collapses, while boxes that are too large cost only a little. If your detector's boxes are tight or
# jittery, pad them by ~5–10% before prompting SAM.

---
# 📝 8 · Recap

**Cheat sheet**
- **Instance segmentation** = a mask per object. **Mask R-CNN** = Faster R-CNN + a per-box 28×28 mask head, trained with per-pixel BCE on the GT class's mask only.
- **RoIAlign** (bilinear, no rounding) instead of RoIPool: essential for pixel accuracy.
- **Mask AP** = 06a's AP with mask IoU. **PQ = SQ × RQ**: outline quality × recognition F1, on non-overlapping segments (things + stuff).
- Resolve overlapping instance masks by painting in score order before panoptic evaluation.
- **SAM / SAM 2**: class-agnostic, promptable (points ± negative points, boxes); heavy encoder once per image, light decoder per prompt. Clicks are ambiguous; boxes much less so.
- **Detect-then-segment** (YOLO → SAM) gives instance masks with no mask training.
- Separate detection errors from mask errors: also evaluate masks given ground-truth boxes.

**Interview-style questions**
1. How does Mask R-CNN extend Faster R-CNN? Why are masks predicted per class but trained only for the GT class?
2. What problem does RoIAlign fix, and why does it matter more for masks than for boxes?
3. Define Panoptic Quality. What do SQ and RQ each tell you?
4. Things vs. stuff: give two examples of each. How does panoptic segmentation treat them?
5. How does SAM handle ambiguous prompts? How do you get the mask you want?
6. Design an instance segmentation system for a new object type with only 50 unlabeled images and no budget for mask labeling.
7. Your model scores high box AP but low mask AP. What does that suggest?

---
# 🎉 Phase 3 complete

You can now detect objects (06) and segment them at the pixel level, per class and per instance (07), and measure every one of those outputs properly.

**Next:** Phase 4, *Transformers & embeddings*: attention and Vision Transformers (08), then DETR (09), which removes anchors and NMS from detection entirely.